# NSE AI Stock Analysis — Model Pipeline

**This notebook is the single source of truth for:**
- Data preprocessing & feature engineering
- LSTM model training & evaluation
- FinBERT sentiment analysis pipeline
- Saving all artefacts (model, scaler, processed datasets)

> ⚠️ **STRICT RULE**: No preprocessing or training logic exists outside this notebook.
> Python modules in `src/` perform inference ONLY using artefacts saved here.

---
**Sections:**
1. Environment Setup
2. Data Loading
3. Data Preprocessing
4. Feature Engineering
5. LSTM Model — Training & Evaluation
6. FinBERT Sentiment Pipeline
7. Artefact Saving


## 1. Environment Setup

In [ ]:
import sys
import os
from pathlib import Path

# ── Resolve project root robustly ──
# If the notebook is inside a 'notebooks/' subdirectory, go one level up.
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# HuggingFace
from transformers import BertTokenizer, BertForSequenceClassification

# Joblib
import joblib

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')
print(f'Project root: {PROJECT_ROOT}')

# Directories
RAW_DIR       = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR     = PROJECT_ROOT / 'models' / 'saved_models'
for d in [RAW_DIR, PROCESSED_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print('Directories ready.')


## 2. Data Loading

In [ ]:
from src.ingestion.data_loader import fetch_nse_data, get_mock_data

TICKER = 'RELIANCE'
PERIOD = '2y'

try:
    df_raw = fetch_nse_data(ticker=TICKER, period=PERIOD, save=True)
    print(f'Live data loaded: {len(df_raw)} rows')
except Exception as e:
    print(f'Live fetch failed ({e}). Using mock data.')
    df_raw = get_mock_data(ticker=TICKER, n_rows=500)

print(df_raw.tail())
print('\nDtypes:')
print(df_raw.dtypes)
print('\nMissing values:')
print(df_raw.isnull().sum())

## 3. Data Preprocessing

In [ ]:
# ── 3.1 Drop duplicates & sort ──────────────────────────
df = df_raw.copy()
df = df[~df.index.duplicated(keep='last')].sort_index()

# ── 3.2 Handle missing values ───────────────────────────
df.ffill(inplace=True)  # forward-fill first
df.bfill(inplace=True)  # back-fill any remaining NaN at start

# ── 3.3 Outlier clipping (IQR method on Close) ──────────
Q1, Q3 = df['Close'].quantile(0.01), df['Close'].quantile(0.99)
df = df[(df['Close'] >= Q1) & (df['Close'] <= Q3)]

print(f'Shape after preprocessing: {df.shape}')

# ── 3.4 Quick EDA plot ───────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(14, 7))
axes[0].plot(df.index, df['Close'], label='Close', color='royalblue')
axes[0].set_title(f'{TICKER} Close Price (cleaned)')
axes[0].legend()
axes[1].bar(df.index, df['Volume'], color='slategray', alpha=0.6)
axes[1].set_title('Volume')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

In [ ]:
import ta

# ── 4.1 RSI ─────────────────────────────────────────────
df['RSI'] = ta.momentum.RSIIndicator(close=df['Close'], window=14).rsi()

# ── 4.2 MACD ────────────────────────────────────────────
macd = ta.trend.MACD(close=df['Close'])
df['MACD']        = macd.macd()
df['MACD_Signal'] = macd.macd_signal()
df['MACD_Hist']   = macd.macd_diff()

# ── 4.3 Bollinger Bands ─────────────────────────────────
bb = ta.volatility.BollingerBands(close=df['Close'], window=20, window_dev=2)
df['BB_Upper'] = bb.bollinger_hband()
df['BB_Lower'] = bb.bollinger_lband()
df['BB_Width'] = bb.bollinger_wband()

# ── 4.4 Volatility (ATR) ────────────────────────────────
df['ATR'] = ta.volatility.AverageTrueRange(
    high=df['High'], low=df['Low'], close=df['Close'], window=14
).average_true_range()

# ── 4.5 Lag features ────────────────────────────────────
for lag in [1, 3, 5]:
    df[f'Close_lag{lag}'] = df['Close'].shift(lag)
    df[f'Volume_lag{lag}'] = df['Volume'].shift(lag)

# ── 4.6 Rolling statistics ──────────────────────────────
df['Close_MA7']  = df['Close'].rolling(7).mean()
df['Close_MA21'] = df['Close'].rolling(21).mean()
df['Close_STD7'] = df['Close'].rolling(7).std()

# ── 4.7 Drop NaN rows created by indicators ─────────────
df.dropna(inplace=True)

print(f'Feature-engineered shape: {df.shape}')
print(df.columns.tolist())

In [ ]:
# ── 4.8 Correlation heatmap ─────────────────────────────
FEATURE_COLS = [
    'Close', 'Open', 'High', 'Low', 'Volume',
    'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower'
]
# Use only the 10 features the LSTM expects
df_feat = df[FEATURE_COLS].copy()

plt.figure(figsize=(10, 8))
sns.heatmap(df_feat.corr(), annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 5. LSTM Model — Training & Evaluation

In [ ]:
# ── 5.1 Normalisation ───────────────────────────────────
WINDOW_SIZE  = 60
TEST_SPLIT   = 0.2

scaler = MinMaxScaler(feature_range=(0, 1))
scaled = scaler.fit_transform(df_feat.values)  # shape: (n, 10)
print(f'Scaled data shape: {scaled.shape}')

# ── 5.2 Sliding-window sequences ────────────────────────
X_list, y_list = [], []
for i in range(WINDOW_SIZE, len(scaled)):
    X_list.append(scaled[i - WINDOW_SIZE:i])   # (60, 10)
    y_list.append(scaled[i, 0])                # Close price (col 0)

X = np.array(X_list)   # (N, 60, 10)
y = np.array(y_list)   # (N,)
print(f'X: {X.shape} | y: {y.shape}')

# ── 5.3 Time-based train/test split ─────────────────────
split_idx = int(len(X) * (1 - TEST_SPLIT))
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

# ── 5.4 PyTorch DataLoaders ─────────────────────────────
BATCH_SIZE = 32

train_ds = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32)
)
test_ds = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.float32)
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)
print(f'Train batches: {len(train_loader)} | Test batches: {len(test_loader)}')

In [ ]:
# ── 5.5 LSTM Architecture ───────────────────────────────
class LSTMModel(nn.Module):
    """Multi-layer LSTM for stock price forecasting."""

    def __init__(self, input_size, hidden_size, num_layers, output_size, dropout=0.2):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])

INPUT_SIZE  = df_feat.shape[1]  # 10
HIDDEN_SIZE = 128
NUM_LAYERS  = 2
OUTPUT_SIZE = 1
DROPOUT     = 0.2

model = LSTMModel(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE, DROPOUT).to(DEVICE)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

In [ ]:
# ── 5.6 Training loop ───────────────────────────────────
EPOCHS    = 50
LR        = 0.001
PATIENCE  = 10

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_counter = 0
best_state = None

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        preds = model(X_batch).squeeze()
        loss = criterion(preds, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        epoch_loss += loss.item()
    avg_train = epoch_loss / len(train_loader)

    # ── Validate ──
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            preds = model(X_batch).squeeze()
            val_loss += criterion(preds, y_batch).item()
    avg_val = val_loss / len(test_loader)
    scheduler.step(avg_val)

    train_losses.append(avg_train)
    val_losses.append(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | Train Loss: {avg_train:.6f} | Val Loss: {avg_val:.6f}')

    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch}.')
        break

# Restore best weights
model.load_state_dict(best_state)
print(f'\nBest validation loss: {best_val_loss:.6f}')

In [ ]:
# ── 5.7 Training curves ─────────────────────────────────
plt.figure(figsize=(12, 4))
plt.plot(train_losses, label='Train Loss', color='royalblue')
plt.plot(val_losses,   label='Val Loss',   color='tomato')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('LSTM Training Curve')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.8 Evaluation ──────────────────────────────────────
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(DEVICE)
        preds = model(X_batch).squeeze().cpu().numpy()
        all_preds.extend(preds if preds.ndim > 0 else [preds])
        all_true.extend(y_batch.numpy())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

# Inverse-scale for interpretable metrics
n_feat = scaler.scale_.shape[0]
dummy = np.zeros((len(all_preds), n_feat))
dummy[:, 0] = all_preds
preds_real = scaler.inverse_transform(dummy)[:, 0]

dummy[:, 0] = all_true
true_real = scaler.inverse_transform(dummy)[:, 0]

rmse = np.sqrt(mean_squared_error(true_real, preds_real))
mae  = mean_absolute_error(true_real, preds_real)
mape = np.mean(np.abs((true_real - preds_real) / true_real)) * 100

print(f'RMSE : {rmse:.2f}')
print(f'MAE  : {mae:.2f}')
print(f'MAPE : {mape:.2f}%')

# Plot predictions vs actual
plt.figure(figsize=(14, 5))
plt.plot(true_real,  label='Actual',    color='royalblue')
plt.plot(preds_real, label='Predicted', color='tomato', alpha=0.8)
plt.title(f'{TICKER} — LSTM Predictions vs Actual (Test Set)')
plt.xlabel('Test Sample Index')
plt.ylabel('Price (INR)')
plt.legend()
plt.tight_layout()
plt.show()

## 6. FinBERT Sentiment Pipeline

In [ ]:
from src.ingestion.data_loader import load_news_data
from src.nlp.finbert_sentiment import load_finbert, predict_sentiment, sentiment_to_signal

news_df = load_news_data()
print(f'News headlines loaded: {len(news_df)}')
print(news_df.head())

In [ ]:
# Load FinBERT (downloads on first run, ~430 MB)
tokenizer, finbert = load_finbert()
print('FinBERT loaded.')

In [ ]:
headlines = news_df['headline'].tolist()
sentiment_results = predict_sentiment(headlines)

sentiment_df = pd.DataFrame(sentiment_results)
print(sentiment_df[['text', 'label', 'score', 'confidence']])

composite = sentiment_to_signal(sentiment_results)
print(f'\nComposite signal score: {composite:+.4f}')

In [ ]:
# Sentiment distribution plot
label_counts = sentiment_df['label'].value_counts()
colors = {'positive': '#2ecc71', 'negative': '#e74c3c', 'neutral': '#95a5a6'}
bar_colors = [colors.get(l, 'grey') for l in label_counts.index]

plt.figure(figsize=(7, 4))
label_counts.plot(kind='bar', color=bar_colors, edgecolor='black')
plt.title('FinBERT Sentiment Distribution')
plt.xlabel('Sentiment')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. Artefact Saving

In [ ]:
# ── 7.1 Save LSTM model ─────────────────────────────────
lstm_path = MODEL_DIR / 'lstm_model.pt'
torch.save(model.state_dict(), lstm_path)
print(f'LSTM model saved → {lstm_path}')

# ── 7.2 Save scaler ─────────────────────────────────────
scaler_path = MODEL_DIR / 'scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f'Scaler saved → {scaler_path}')

# ── 7.3 Save processed dataset ──────────────────────────
proc_path = PROCESSED_DIR / 'processed_stock_data.csv'
df_feat.to_csv(proc_path)
print(f'Processed dataset saved → {proc_path}')

# ── 7.4 Save sentiment output ───────────────────────────
sent_path = PROCESSED_DIR / 'sentiment_output.csv'
sentiment_df.to_csv(sent_path, index=False)
print(f'Sentiment output saved → {sent_path}')

print('\n✅ All artefacts saved. Python modules are ready to use.')

In [ ]:
# ── 7.5 Verification: test loading artefacts ────────────
import joblib, torch

loaded_scaler = joblib.load(scaler_path)
assert loaded_scaler is not None, 'Scaler load failed'

test_model = LSTMModel(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, OUTPUT_SIZE, DROPOUT)
test_model.load_state_dict(torch.load(lstm_path, map_location='cpu', weights_only=True))
test_model.eval()
dummy_input = torch.randn(1, WINDOW_SIZE, INPUT_SIZE)
dummy_output = test_model(dummy_input)
assert dummy_output.shape == (1, 1), 'Model output shape mismatch'

print('✅ All artefacts verified successfully.')
print(f'   Model output for dummy input: {dummy_output.item():.6f}')